# Boosting — dos PDF de Moodle, 8 de octubre

Objetivo: reproducir AdaBoost, Gradient Boosting y XGBoost y explicar pesos, residuos, regularización y evaluación. [Guía completa](README.md) y [PDF principal](../../materialak/5072_2_07_Boosting.pdf).

Ejecutar todas las celdas desde esta carpeta con el kernel `.venv/bin/python` creado por `uv sync --frozen`. Datos sklearn y ejemplos manuales del PDF; no hay descarga, llamadas API ni cargas pickle. Se sobrescriben solo `resultados/emaitzak.json` y `resultados/boosting.png`.

La accuracy de entrenamiento no estima generalización; el ejemplo diabetes del PDF usa el mismo conjunto en fit y eval_set. La extensión separa 353/89 filas, sin elegir parámetros con test. Es una división educativa, no validación clínica ni prueba de superioridad general de un modelo.

## AdaBoost: las dos rondas del ejemplo manual, sin redondeo intermedio.

In [1]:
import json
from importlib.metadata import version
from pathlib import Path

import matplotlib
import numpy as np
from sklearn.datasets import (
    load_diabetes,
    load_iris,
    make_classification,
    make_regression,
)
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier, XGBRegressor

matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / "boosting.py").is_file(), "Ejecutar desde la carpeta Boosting"


def adaboost_manual():
    """Reproduce las reglas elegidas en el PDF; no afirma que sean óptimas."""
    y = np.array([1, 1, -1, 1])
    predictions = [np.array([1, 1, -1, -1]), np.array([-1, -1, -1, 1])]
    weights = np.full(4, 0.25)
    margin = np.zeros(4)
    rounds = []
    for pred in predictions:
        error = float(weights[pred != y].sum())
        alpha = float(0.5 * np.log((1 - error) / error))
        weights *= np.exp(-alpha * y * pred)
        weights /= weights.sum()
        margin += alpha * pred
        rounds.append({"error": error, "alpha": alpha, "weights": weights.tolist()})
    return {
        "rounds": rounds,
        "margin": margin.tolist(),
        "prediction": np.sign(margin).astype(int).tolist(),
        "accuracy_train": float(np.mean(np.sign(margin) == y)),
    }

## Gradient Boosting manual: media, residuos y dos stumps fijados.

In [2]:
def gradient_manual():
    """Pérdida cuadrática; para otras pérdidas se usan gradientes negativos."""
    y = np.array([30.0, 32.0, 48.0, 50.0])
    prediction = np.full(4, y.mean())
    stages = [
        {
            "prediction": prediction.tolist(),
            "residual": (y - prediction).tolist(),
            "mse": float(mean_squared_error(y, prediction)),
        }
    ]
    for _ in range(2):
        residual = y - prediction
        stump = np.repeat([residual[:2].mean(), residual[2:].mean()], 2)
        prediction += 0.5 * stump
        stages.append(
            {
                "prediction": prediction.tolist(),
                "residual": (y - prediction).tolist(),
                "mse": float(mean_squared_error(y, prediction)),
            }
        )
    return stages

## XGBoost manual: separar la puntuación docente de la ganancia del objetivo.

In [3]:
def xgboost_manual():
    """Gradiente pred-y, Hessiano 1; lambda=1, eta=.3 y base_score=40."""
    residual = np.array([-10.0, -8.0, 8.0, 10.0])

    def score(values):
        return float(values.sum() ** 2 / (len(values) + 1))

    scores = {
        str(split): score(residual[:split]) + score(residual[split:]) - score(residual)
        for split in (1, 2, 3)
    }
    outputs = np.repeat([residual[:2].sum() / 3, residual[2:].sum() / 3], 2)
    prediction = 40 + 0.3 * outputs
    return {
        "score_improvement_pdf": scores,
        "objective_improvement_half_squared": scores["2"] / 2,
        "leaf_output": outputs.tolist(),
        "prediction": prediction.tolist(),
        "residual": (np.array([30, 32, 48, 50]) - prediction).tolist(),
    }

## Cuatro ejemplos del PDF principal: conservar datos y divisiones.

In [4]:
def examples_pdf():
    X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    ada = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=42
    )
    ada.fit(X_train, y_train)
    dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
    ada_result = {
        "accuracy_test": float(ada.score(X_test, y_test)),
        "baseline_accuracy_test": float(dummy.score(X_test, y_test)),
        "n_train": len(y_train),
        "n_test": len(y_test),
    }

    X, y = make_regression(random_state=0)
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
    gb = GradientBoostingRegressor(random_state=0).fit(X_train, y_train)
    baseline = DummyRegressor().fit(X_train, y_train)
    gb_result = {
        "r2_test": float(gb.score(X_test, y_test)),
        "mae_test": float(mean_absolute_error(y_test, gb.predict(X_test))),
        "baseline_r2_test": float(baseline.score(X_test, y_test)),
        "n_train": len(y_train),
        "n_test": len(y_test),
    }

    X, y = load_iris(return_X_y=True)
    # El PDF no estratifica. Se conserva su split; no se seleccionan parámetros con test.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    clf = XGBClassifier(n_jobs=2, random_state=42).fit(X_train, y_train)
    iris_result = {
        "accuracy_test": float(accuracy_score(y_test, clf.predict(X_test))),
        "n_train": len(y_train),
        "n_test": len(y_test),
    }

    X, y = load_diabetes(return_X_y=True)
    # Literal del PDF: eval_set es entrenamiento, no validación fuera de muestra.
    reg = XGBRegressor(
        tree_method="hist", eval_metric=mean_absolute_error, n_jobs=2, random_state=0
    )
    reg.fit(X, y, eval_set=[(X, y)], verbose=False)
    diabetes_train = {
        "mae_train": float(mean_absolute_error(y, reg.predict(X))),
        "r2_train": float(reg.score(X, y)),
        "n_train": len(y),
        "n_test": 0,
        "eval_set_is_train": True,
    }
    return {
        "adaboost": ada_result,
        "gradient_boosting": gb_result,
        "xgboost_iris": iris_result,
        "xgboost_diabetes_literal": diabetes_train,
    }

## Extensión: test diabetes separado, baseline y parámetros definidos antes.

In [5]:
def diabetes_holdout():
    X, y = load_diabetes(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    models = {
        "baseline": DummyRegressor(),
        "gradient_boosting": GradientBoostingRegressor(random_state=42),
        "xgboost": XGBRegressor(tree_method="hist", n_jobs=2, random_state=42),
    }
    result = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        prediction = model.predict(X_test)
        result[name] = {
            "mae_test": float(mean_absolute_error(y_test, prediction)),
            "r2_test": float(r2_score(y_test, prediction)),
            "n_train": len(y_train),
            "n_test": len(y_test),
        }
    return result

## Comprobar una ronda manual con la biblioteca, lambda/eta y pruning.

In [6]:
def xgboost_one_tree():
    X = np.arange(1, 5, dtype=float).reshape(-1, 1)
    y = np.array([30.0, 32.0, 48.0, 50.0])
    result = {}
    for gamma in (0, 100, 250):
        model = XGBRegressor(
            n_estimators=1,
            max_depth=1,
            reg_lambda=1,
            learning_rate=0.3,
            gamma=gamma,
            base_score=40,
            tree_method="hist",
            n_jobs=2,
            random_state=0,
        )
        model.fit(X, y)
        result[str(gamma)] = {
            "prediction": model.predict(X).tolist(),
            "tree": json.loads(
                model.get_booster().get_dump(dump_format="json", with_stats=True)[0]
            ),
        }
    return result


def main():
    results = {
        "versions": {
            name: version(name) for name in ("numpy", "scikit-learn", "xgboost-cpu")
        },
        "manual_ada": adaboost_manual(),
        "manual_gb": gradient_manual(),
        "manual_xgb": xgboost_manual(),
        "pdf_examples": examples_pdf(),
        "diabetes_holdout_extension": diabetes_holdout(),
        "xgb_one_tree": xgboost_one_tree(),
    }
    assert np.allclose(
        results["manual_ada"]["rounds"][0]["weights"], [1 / 6, 1 / 6, 1 / 6, 1 / 2]
    )
    assert np.allclose(
        results["manual_gb"][2]["prediction"], [33.25, 33.25, 46.75, 46.75]
    )
    assert np.allclose(
        results["xgb_one_tree"]["0"]["prediction"], [38.2, 38.2, 41.8, 41.8]
    )
    assert np.allclose(results["xgb_one_tree"]["250"]["prediction"], [40, 40, 40, 40])
    output = ROOT / "resultados"
    output.mkdir(exist_ok=True)
    (output / "emaitzak.json").write_text(
        json.dumps(results, indent=2, allow_nan=False) + "\n"
    )
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    stages = results["manual_gb"]
    axes[0].plot([0, 1, 2], [s["mse"] for s in stages], marker="o")
    axes[0].set(
        xlabel="Iteración GB manual",
        ylabel="MSE entrenamiento",
        title="Reducción observada, cuatro puntos",
    )
    holdout = results["diabetes_holdout_extension"]
    axes[1].bar(list(holdout), [v["mae_test"] for v in holdout.values()])
    axes[1].set(
        ylabel="MAE test diabetes", title="Una división reservada; menor es mejor"
    )
    axes[1].tick_params(axis="x", labelrotation=20)
    fig.tight_layout()
    fig.savefig(output / "boosting.png", dpi=150)
    plt.close(fig)
    print(json.dumps(results, indent=2, allow_nan=False))
    return results


if __name__ == "__main__":
    main()

{
  "versions": {
    "numpy": "2.5.3",
    "scikit-learn": "1.9.1",
    "xgboost-cpu": "3.4.1"
  },
  "manual_ada": {
    "rounds": [
      {
        "error": 0.25,
        "alpha": 0.5493061443340549,
        "weights": [
          0.16666666666666666,
          0.16666666666666666,
          0.16666666666666666,
          0.5000000000000001
        ]
      },
      {
        "error": 0.3333333333333333,
        "alpha": 0.34657359027997275,
        "weights": [
          0.25,
          0.25,
          0.12499999999999999,
          0.37500000000000006
        ]
      }
    ],
    "margin": [
      0.20273255405408214,
      0.20273255405408214,
      -0.8958797346140277,
      -0.20273255405408214
    ],
    "prediction": [
      1,
      1,
      -1,
      -1
    ],
    "accuracy_train": 0.75
  },
  "manual_gb": [
    {
      "prediction": [
        40.0,
        40.0,
        40.0,
        40.0
      ],
      "residual": [
        -10.0,
        -8.0,
        8.0,
        10.0
  

## Interpretación

AdaBoost manual: error inicial 1/4, alpha=ln(3)/2; después pesos (1/6,1/6,1/6,1/2). El segundo stump tiene error 1/3 y alpha=ln(2)/2: P4 sigue mal tras ambos votos.

Gradient Boosting: MSE de los cuatro puntos baja 82 → 21.25 → 6.0625. Esto describe entrenamiento sobre los puntos construidos.

XGBoost: lambda reduce las hojas a ±6 y eta=.3 lleva predicciones a 38.2/41.8. El score docente mejora 216; el objetivo teórico con media pérdida cuadrática mejora 108 antes de gamma. El dump real registra gain=216. No mezclar escalas de fórmulas y umbrales. Los tests verifican directamente gamma=100 y 250 en la biblioteca instalada.

Consultar JSON y guía para resultados actuales y versiones. No se realizaron entregas Moodle.